In [2]:
!pip install boto3

     |████████████████████████████████| 139 kB 14.4 MB/s            
     |████████████████████████████████| 12.7 MB 41.2 MB/s            
     |████████████████████████████████| 82 kB 552 kB/s             


# Code final

In [5]:
import boto3
from botocore.client import Config
import re

def create_s3_client(endpoint_url="http://minio:9000", access_key="minio", secret_key="minio123"):
    """
    Creates and returns an S3 client with the given parameters.
    """
    return boto3.client(
        's3',
        endpoint_url=endpoint_url,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        config=Config(signature_version='s3v4')
    )

def get_part_day_values(s3_client, bucket_name, prefix=""):
    """
    Retrieves 'part_day' values from files under the given bucket path.
    The files are named with a pattern like 'YYYY-MM-DD.success'.
    
    Args:
        s3_client: The S3 client object.
        bucket_name: The name of the S3 bucket.
        prefix: The prefix path in the S3 bucket.
    
    Returns:
        A sorted list of unique 'part_day' values (dates in 'YYYY-MM-DD' format).
    """
    date_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})\.success")
    part_day_values = set()
    
    # Use paginator to retrieve all objects under the specified bucket and prefix
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            # Extract the date from file name if it matches the pattern YYYY-MM-DD.success
            match = date_pattern.search(obj["Key"])
            if match:
                part_day_values.add(match.group(1))  # Add the date to the set

    return sorted(part_day_values)

def list_unique_dates(s3_client, bucket_name, prefix=""):
    """
    Lists unique date values (YYYY-MM-DD) from S3 keys in the specified bucket and prefix.
    """
    date_pattern = re.compile(r"\d{4}-\d{2}-\d{2}")
    unique_dates = set()
    paginator = s3_client.get_paginator("list_objects_v2")
    
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            match = date_pattern.search(obj["Key"])  # Assigning match without walrus operator
            if match:  # Checking if match is not None
                unique_dates.add(match.group(0))

    return sorted(unique_dates)

def build_part_day_todo():
    s3_client = create_s3_client()
    bucket_name = "velib"

    # Retrieve part_day_done
    part_day_done = get_part_day_values(s3_client, bucket_name, "_tech/flags/job_success_silver")
    print("part_day_done =", part_day_done)

    # Retrieve unique dates
    part_day_all = list_unique_dates(s3_client, bucket_name, "bronze")
    print("part_day_all =", part_day_all)

    # Calculate part_day_todo
    part_day_todo = [item for item in part_day_all if item not in part_day_done]
    print("part_day_todo =", part_day_todo)
    
    return part_day_todo

build_part_day_todo()

part_day_done = ['2024-10-30']
part_day_all = ['2024-10-29', '2024-10-30', '2024-10-31']
part_day_todo = ['2024-10-29', '2024-10-31']


['2024-10-29', '2024-10-31']

In [9]:
def write_object_to_minio(s3_client, bucket_name, object_key, data):
    """
    Writes an object to MinIO.

    Args:
        s3_client: The S3 client object.
        bucket_name: The name of the S3 bucket.
        object_key: The key (filename) for the object to be written.
        data: The data to be written (can be a string or bytes).
    """
    # If data is a string, encode it to bytes
    if isinstance(data, str):
        data = data.encode('utf-8')
    
    # Upload the object to the specified bucket and key
    s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=data)
    print(f"Successfully uploaded {object_key} to bucket {bucket_name}.")
    
s3_client = create_s3_client()
write_object_to_minio(s3_client, "velib", "_tech/flags/job_success_silver/"+'2024-10-30.success', "content")

Successfully uploaded _tech/flags/job_success_silver/2024-10-30.success to bucket velib.
